# Bounding Box Review

Review and filter detections per panel **before** running `extract_crops.py`.

**Workflow:**
1. Pick a panel from the dropdown  
2. Adjust the containment threshold — sub-crops (bboxes mostly inside a larger one) are auto-excluded (red)  
3. Toggle individual detections in the card grid to manually include/exclude  
4. Click **Save approved** — writes `annotated/<panel>_approved.json` (same schema, excluded detections removed)  
5. Run `extract_crops.py --approved-dir` to crop only approved detections

Approved files accumulate — you can review panels in any order.

In [4]:
# ── Cell 1: imports & paths ────────────────────────────────────────────────
import json
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

REPO_ROOT    = Path("../..")
ANNOTATED    = REPO_ROOT / "frobenius_artifacts/analysis/annotated"
PANELS_DIR   = REPO_ROOT / "frobenius_artifacts/analysis/panels"

# Collect all panels that have both a detections JSON and a panel PNG
_jsons = sorted(ANNOTATED.glob("*_detections.json"))
_panels = []
for j in _jsons:
    stem = j.stem.replace("_detections", "")
    png  = PANELS_DIR / f"{stem}.png"
    if png.exists():
        _panels.append((stem, j, png))
    else:
        # try _cropped suffix variant
        alt = PANELS_DIR / f"{stem}_cropped.png"
        if alt.exists():
            _panels.append((stem, j, alt))

print(f"{len(_panels)} panels with both detections JSON and panel PNG")
for stem, j, png in _panels[:5]:
    dets = json.loads(j.read_text())
    print(f"  {stem}: {len(dets)} detections")

57 panels with both detections JSON and panel PNG
  EBA-B_00425_Ibadan_q97912_i1_panel_00: 3 detections
  EBA-B_00642_q98212_i3_panel_00: 8 detections
  EBA-Div_00302_q166558_i1_panel_00: 3 detections
  EBA-Div_00303_Ado_Ekiti_q166559_i1_panel_00: 1 detections
  EBA-Div_00311_Ife_q166566_i1_panel_00: 6 detections


In [8]:
## ── Cell 2: main review UI ─────────────────────────────────────────────────
#
# Controls:
#   Panel dropdown       — pick a panel to review
#   Containment / IoU    — auto-exclusion filters
#   Detection cards      — toggle include/exclude per detection
#   Draw bbox sliders    — x/y/w/h sliders auto-bounded to panel size; cyan
#                          preview updates live on the image as you drag
#   [Add bbox]           — commit the slider bbox as source="manual"
#   [Show candidates]    — overlay SAM masks below threshold as dashed yellow
#   Save approved        — writes <panel>_approved.json with source field

import io as _io

# ── Helpers ───────────────────────────────────────────────────────────────────
def _containment(a, b):
    ax1, ay1 = a["x"], a["y"];  ax2, ay2 = ax1 + a["w"], ay1 + a["h"]
    bx1, by1 = b["x"], b["y"];  bx2, by2 = bx1 + b["w"], by1 + b["h"]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1: return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    min_area = min(a["w"] * a["h"], b["w"] * b["h"])
    return inter / min_area if min_area > 0 else 0.0

def _iou_bbox(a, b):
    ax1, ay1 = a["x"], a["y"];  ax2, ay2 = ax1 + a["w"], ay1 + a["h"]
    bx1, by1 = b["x"], b["y"];  bx2, by2 = bx1 + b["w"], by1 + b["h"]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1: return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    union = a["w"] * a["h"] + b["w"] * b["h"] - inter
    return inter / union if union > 0 else 0.0

def _auto_exclude(dets, threshold):
    n = len(dets)
    areas = [d["bbox"]["w"] * d["bbox"]["h"] for d in dets]
    by_size = sorted(range(n), key=lambda i: -areas[i])
    suppress = set()
    for pos, i in enumerate(by_size):
        if i in suppress: continue
        for j in by_size[pos + 1:]:
            if j not in suppress and _containment(dets[i]["bbox"], dets[j]["bbox"]) >= threshold:
                suppress.add(j)
    return suppress

def _thumb(panel_img, bbox, size=96, bg=(30, 30, 30)):
    iw, ih = panel_img.size
    b = bbox
    crop = panel_img.crop((max(0, b["x"]), max(0, b["y"]),
                           min(iw, b["x"] + b["w"]), min(ih, b["y"] + b["h"])))
    crop.thumbnail((size, size))
    sq = Image.new("RGB", (size, size), bg)
    sq.paste(crop, ((size - crop.width) // 2, (size - crop.height) // 2))
    buf = _io.BytesIO();  sq.save(buf, format="PNG")
    return widgets.Image(value=buf.getvalue(), format="png", width=size, height=size)


# ── Panel image drawing ───────────────────────────────────────────────────────
MAX_DISPLAY_W = 900

def _draw_panel(panel_img, dets, include_mask, out,
                candidates=None, pending_bbox=None):
    """Render panel.  pending_bbox (cyan) = live slider preview before Add."""
    iw, ih = panel_img.size
    scale  = min(1.0, MAX_DISPLAY_W / iw)
    dw, dh = int(iw * scale), int(ih * scale)

    fig, ax = plt.subplots(figsize=(dw / 96, dh / 96), dpi=96)
    ax.imshow(panel_img.resize((dw, dh), Image.LANCZOS))
    ax.axis("off")

    for i, d in enumerate(dets):
        b = d["bbox"]
        x, y, w, h = b["x"]*scale, b["y"]*scale, b["w"]*scale, b["h"]*scale
        inc   = include_mask[i]
        color = "#44dd44" if inc else "#ff4444"
        ax.add_patch(mpatches.Rectangle(
            (x, y), w, h,
            linewidth=2.5 if inc else 1.5,
            edgecolor=color, facecolor="none",
            alpha=0.9 if inc else 0.55,
        ))
        ax.text(x+3, y+3, str(d["index"]),
                fontsize=max(6, int(10*scale)), color=color, va="top", fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.4, pad=1, linewidth=0))

    if candidates:
        for c in candidates:
            b = c["bbox"]
            x, y, w, h = b["x"]*scale, b["y"]*scale, b["w"]*scale, b["h"]*scale
            ax.add_patch(mpatches.Rectangle(
                (x, y), w, h,
                linewidth=1.5, edgecolor="#ffcc00", facecolor="none",
                alpha=0.6, linestyle="dashed",
            ))
            ax.text(x+3, y+3, f"C{c['_cand_idx']}",
                    fontsize=max(5, int(8*scale)), color="#ffcc00", va="top",
                    bbox=dict(facecolor="black", alpha=0.3, pad=1, linewidth=0))

    # ── Cyan live-preview rect for pending manual bbox ────────────────────────
    if pending_bbox and pending_bbox["w"] > 0 and pending_bbox["h"] > 0:
        b = pending_bbox
        x, y, w, h = b["x"]*scale, b["y"]*scale, b["w"]*scale, b["h"]*scale
        # filled tint + solid border
        ax.add_patch(mpatches.Rectangle(
            (x, y), w, h,
            linewidth=0, facecolor="#00ffff", alpha=0.12,
        ))
        ax.add_patch(mpatches.Rectangle(
            (x, y), w, h,
            linewidth=2.5, edgecolor="#00ffff", facecolor="none",
            alpha=0.95, linestyle=(0, (6, 3)),   # long dash
        ))
        ax.text(x+4, y+4, "new",
                fontsize=max(7, int(11*scale)), color="#00ffff", va="top",
                fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.45, pad=1, linewidth=0))

    plt.tight_layout(pad=0)
    out.clear_output(wait=True)
    with out: plt.show()
    plt.close(fig)


# ── State ─────────────────────────────────────────────────────────────────────
_state = {
    "stem": None, "dets": [], "candidates": [], "panel_img": None,
    "checkboxes": [], "cand_checkboxes": [],
    "auto_excl": set(), "dirty": False, "show_candidates": False,
}

# ── Widgets ───────────────────────────────────────────────────────────────────
_sl = dict(continuous_update=False,
           style={"description_width": "180px"},
           layout=widgets.Layout(width="60%"))

w_panel = widgets.Dropdown(
    options=[(s, i) for i, (s, _, __) in enumerate(_panels)],
    description="Panel:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="90%"),
)
w_contain = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.75,
    description="containment threshold", readout_format=".2f", **_sl)
w_min_iou = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.0,
    description="min predicted_iou", readout_format=".2f", **_sl)

btn_save    = widgets.Button(description="Save approved", button_style="success",
                             layout=widgets.Layout(width="160px", margin="8px 0"))
btn_all_in  = widgets.Button(description="Include all",   button_style="info",
                             layout=widgets.Layout(width="120px"))
btn_all_out = widgets.Button(description="Exclude all",   button_style="warning",
                             layout=widgets.Layout(width="120px"))
btn_reset   = widgets.Button(description="Reset to auto", button_style="",
                             layout=widgets.Layout(width="130px"))
btn_candidates = widgets.Button(description="Show candidates", button_style="",
                                layout=widgets.Layout(width="155px"))

# ── Draw-bbox sliders (Phase 1c) ──────────────────────────────────────────────
# Bounded to panel size; max is updated in _reset_draw_bounds() on panel load.
_draw_sl = dict(
    continuous_update=True,    # live preview as you drag
    style={"description_width": "25px"},
    layout=widgets.Layout(width="100%"),
)
w_mx = widgets.IntSlider(value=0,   min=0, max=2000, step=1,  description="x", **_draw_sl)
w_my = widgets.IntSlider(value=0,   min=0, max=2000, step=1,  description="y", **_draw_sl)
w_mw = widgets.IntSlider(value=200, min=1, max=2000, step=1,  description="w", **_draw_sl)
w_mh = widgets.IntSlider(value=200, min=1, max=2000, step=1,  description="h", **_draw_sl)

# Integer readout labels beside each slider value
_lbl_mx = widgets.Label(); _lbl_my = widgets.Label()
_lbl_mw = widgets.Label(); _lbl_mh = widgets.Label()

def _sync_labels(_=None):
    _lbl_mx.value = str(w_mx.value); _lbl_my.value = str(w_my.value)
    _lbl_mw.value = str(w_mw.value); _lbl_mh.value = str(w_mh.value)

_sync_labels()

btn_add_manual = widgets.Button(description="Add bbox ✓", button_style="info",
                                layout=widgets.Layout(width="110px"))

_W = widgets.Layout(width="46%")
draw_section = widgets.VBox([
    widgets.HTML("<b style='font-size:12px;color:#00cccc'>Draw manual bbox "
                 "— drag sliders; cyan rect updates live on the image</b>"),
    widgets.HBox([
        widgets.VBox([
            widgets.HBox([widgets.Label("x", layout=widgets.Layout(width="14px")),
                          w_mx, _lbl_mx], layout=_W),
            widgets.HBox([widgets.Label("y", layout=widgets.Layout(width="14px")),
                          w_my, _lbl_my], layout=_W),
        ], layout=widgets.Layout(width="50%")),
        widgets.VBox([
            widgets.HBox([widgets.Label("w", layout=widgets.Layout(width="14px")),
                          w_mw, _lbl_mw], layout=_W),
            widgets.HBox([widgets.Label("h", layout=widgets.Layout(width="14px")),
                          w_mh, _lbl_mh], layout=_W),
        ], layout=widgets.Layout(width="50%")),
    ]),
    btn_add_manual,
], layout=widgets.Layout(
    border="1px solid #006666", padding="8px 10px", margin="6px 0",
    background="#001a1a",
))

out_panel  = widgets.Output()
out_cards  = widgets.Output()
out_cands  = widgets.Output()
out_status = widgets.Output()


# ── Dirty state ───────────────────────────────────────────────────────────────
def _mark_dirty():
    _state["dirty"] = True
    btn_save.description = "Save approved ●"
    btn_save.button_style = "warning"

def _mark_clean():
    _state["dirty"] = False
    btn_save.description = "Save approved"
    btn_save.button_style = "success"

def _include_mask():
    return [cb.value for cb in _state["checkboxes"]]

def _pending_bbox():
    return {"x": w_mx.value, "y": w_my.value, "w": w_mw.value, "h": w_mh.value}

def _redraw_panel(with_pending=False):
    if _state["panel_img"] is None: return
    cands   = _state["candidates"] if _state["show_candidates"] else None
    pending = _pending_bbox() if with_pending else None
    _draw_panel(_state["panel_img"], _state["dets"], _include_mask(),
                out_panel, candidates=cands, pending_bbox=pending)

def _on_det_change(change):
    _mark_dirty()
    _redraw_panel()

def _on_draw_change(change):
    """Live preview: redraw with cyan pending rect as sliders move."""
    _sync_labels()
    _redraw_panel(with_pending=True)


# ── Detection cards ───────────────────────────────────────────────────────────
def _build_cards():
    dets      = _state["dets"]
    panel_img = _state["panel_img"]
    auto_excl = _state["auto_excl"]
    checkboxes, cards = [], []
    THUMB = 96

    for i, d in enumerate(dets):
        tw = _thumb(panel_img, d["bbox"], THUMB)
        auto_tag = " ⚠ sub-crop" if i in auto_excl else ""
        src_tag  = f" [{d['source']}]" if "source" in d else ""
        cb = widgets.Checkbox(value=(i not in auto_excl), description="Include",
                              indent=False, style={"description_width": "initial"},
                              layout=widgets.Layout(width="100px"))
        checkboxes.append(cb)
        cb.observe(_on_det_change, names="value")

        iou  = d.get("predicted_iou", d.get("pred_iou", "?"))
        stab = d.get("stability_score", "?")
        area = d.get("area_ratio", "?")
        b    = d["bbox"]
        area_s = f"{area:.3f}" if isinstance(area, float) else str(area)
        iou_s  = f"{iou:.3f}"  if isinstance(iou,  float) else str(iou)
        stab_s = f"{stab:.3f}" if isinstance(stab, float) else str(stab)
        meta = widgets.HTML(
            f"<div style='font-size:11px;line-height:1.5;color:#ccc'>"
            f"<b>#{d['index']}</b> {d.get('scale','?')}{src_tag}<br>"
            f"area: {area_s}<br>"
            f"iou: {iou_s}<br>"
            f"stab: {stab_s}<br>"
            f"{b['w']}×{b['h']} px"
            f"<span style='color:#ff8888'>{auto_tag}</span></div>"
        )
        cards.append(widgets.VBox(
            [tw, meta, cb],
            layout=widgets.Layout(border="1px solid #333", padding="4px",
                                  margin="3px", width="120px", background="#1a1a1a"),
        ))

    _state["checkboxes"] = checkboxes
    rows = [widgets.HBox(cards[r:r+6]) for r in range(0, len(cards), 6)]
    out_cards.clear_output(wait=True)
    with out_cards:
        display(widgets.VBox(rows) if rows else widgets.HTML("<i>No detections</i>"))


# ── Candidate cards ───────────────────────────────────────────────────────────
def _build_candidate_cards():
    cands = _state["candidates"]
    if not cands:
        out_cands.clear_output(wait=True)
        with out_cands:
            display(widgets.HTML(
                "<i style='color:#888'>No candidate pool — re-run motif_segment.py "
                "to generate _detections_raw.json</i>"))
        return

    cand_checkboxes, cards = [], []
    THUMB = 96
    for c in cands:
        tw = _thumb(_state["panel_img"], c["bbox"], THUMB, bg=(20, 15, 5))
        cb = widgets.Checkbox(value=False, description="Promote", indent=False,
                              style={"description_width": "initial"},
                              layout=widgets.Layout(width="100px"))
        cand_checkboxes.append(cb)
        cb.observe(_on_det_change, names="value")
        iou  = c.get("predicted_iou", "?");  area = c.get("area_ratio", "?")
        b    = c["bbox"]
        area_s = f"{area:.3f}" if isinstance(area, float) else str(area)
        iou_s  = f"{iou:.3f}"  if isinstance(iou,  float) else str(iou)
        meta = widgets.HTML(
            f"<div style='font-size:11px;line-height:1.5;color:#cc9900'>"
            f"<b>C{c['_cand_idx']}</b> candidate<br>"
            f"area: {area_s}<br>"
            f"iou: {iou_s}<br>"
            f"{b['w']}×{b['h']} px</div>"
        )
        cards.append(widgets.VBox(
            [tw, meta, cb],
            layout=widgets.Layout(border="1px solid #554400", padding="4px",
                                  margin="3px", width="120px", background="#1a0d00"),
        ))

    _state["cand_checkboxes"] = cand_checkboxes
    rows = [widgets.HBox(cards[r:r+6]) for r in range(0, len(cards), 6)]
    out_cands.clear_output(wait=True)
    with out_cands:
        display(widgets.VBox([
            widgets.HTML(f"<b style='color:#ffcc00'>Candidate pool — "
                         f"{len(cands)} masks not in filtered detections</b>"),
            widgets.VBox(rows),
        ]))

def _load_candidates():
    raw_path = ANNOTATED / f"{_state['stem']}_detections_raw.json"
    if not raw_path.exists():
        _state["candidates"] = []; return
    raw = json.loads(raw_path.read_text())
    cur = [d["bbox"] for d in _state["dets"]]
    cands, idx = [], 0
    for m in raw:
        if not any(_iou_bbox(m["bbox"], cb) > 0.3 for cb in cur):
            m["_cand_idx"] = idx;  cands.append(m);  idx += 1
    _state["candidates"] = cands


# ── Draw-bounds reset (call on every panel load) ──────────────────────────────
def _reset_draw_bounds():
    """Set slider max values to current panel dimensions; reset to safe values."""
    panel_img = _state["panel_img"]
    if not panel_img: return
    pw, ph = panel_img.size
    # Reset values before expanding max to avoid out-of-range complaints
    w_mx.value = 0;  w_my.value = 0
    w_mw.value = min(200, pw);  w_mh.value = min(200, ph)
    w_mx.max = pw - 1;  w_my.max = ph - 1
    w_mw.max = pw;      w_mh.max = ph
    _sync_labels()


# ── Panel loading ─────────────────────────────────────────────────────────────
def _load_panel(_=None):
    if _state["dirty"] and _state["stem"] is not None:
        _save(autosave=True)

    idx = w_panel.value
    stem, json_path, png_path = _panels[idx]
    approved_path = ANNOTATED / f"{stem}_approved.json"
    if approved_path.exists():
        dets = json.loads(approved_path.read_text());  _src = "approved"
    else:
        dets = json.loads(json_path.read_text());      _src = "detections"

    panel_img = Image.open(png_path).convert("RGB")
    min_iou   = w_min_iou.value
    dets      = [d for d in dets
                 if d.get("predicted_iou", d.get("pred_iou", 1.0)) >= min_iou]
    auto_excl = _auto_exclude(dets, w_contain.value)

    _state.update(stem=stem, dets=dets, panel_img=panel_img,
                  auto_excl=auto_excl, candidates=[], cand_checkboxes=[])
    _mark_clean()
    _reset_draw_bounds()
    _build_cards()
    _redraw_panel()
    out_cands.clear_output()
    out_status.clear_output()
    with out_status:
        print(f"{stem} — {len(dets)} detections, "
              f"{len(auto_excl)} auto-excluded  [from {_src}]")
        print(f"Panel size: {panel_img.size[0]}×{panel_img.size[1]} px")

    if _state["show_candidates"]:
        _load_candidates();  _build_candidate_cards();  _redraw_panel()


def _on_threshold(_=None):
    dets = _state["dets"]
    if not dets: return
    prev = _include_mask()
    _state["auto_excl"] = _auto_exclude(dets, w_contain.value)
    _build_cards()
    for i, cb in enumerate(_state["checkboxes"]):
        cb.value = prev[i] if i < len(prev) else (i not in _state["auto_excl"])
    _redraw_panel()

def _include_all(_):
    for cb in _state["checkboxes"]: cb.value = True
    _mark_dirty()

def _exclude_all(_):
    for cb in _state["checkboxes"]: cb.value = False
    _mark_dirty()

def _reset_to_auto(_):
    auto = _state["auto_excl"]
    for i, cb in enumerate(_state["checkboxes"]): cb.value = (i not in auto)
    _mark_dirty()

def _on_candidates_toggle(_):
    _state["show_candidates"] = not _state["show_candidates"]
    if _state["show_candidates"]:
        btn_candidates.description = "Hide candidates"
        btn_candidates.button_style = "warning"
        _load_candidates();  _build_candidate_cards()
    else:
        btn_candidates.description = "Show candidates"
        btn_candidates.button_style = ""
        out_cands.clear_output()
    _redraw_panel()

def _on_add_manual(_):
    """Commit the current slider position as a manual detection."""
    pb = _pending_bbox()
    if pb["w"] <= 0 or pb["h"] <= 0:
        out_status.clear_output()
        with out_status: print("  ⚠ w and h must be > 0"); return

    panel_img = _state["panel_img"]
    if panel_img is None: return

    pw, ph = panel_img.size
    area_ratio = (pb["w"] * pb["h"]) / (pw * ph) if pw * ph > 0 else 0.0
    prev_mask  = _include_mask()
    new_idx    = len(_state["dets"])

    _state["dets"].append({
        "index": new_idx, "bbox": pb,
        "scale": "motif" if area_ratio < 0.25 else "register",
        "area_ratio": round(area_ratio, 5),
        "predicted_iou": 1.0, "stability_score": 1.0,
        "source": "manual",
    })
    _build_cards()
    for i, cb in enumerate(_state["checkboxes"]):
        if i < len(prev_mask): cb.value = prev_mask[i]
    _redraw_panel()   # no pending — it's been committed
    _mark_dirty()
    out_status.clear_output()
    with out_status:
        print(f"  Added manual bbox #{new_idx}: "
              f"({pb['x']},{pb['y']}) {pb['w']}×{pb['h']}")

def _save(btn_event=None, autosave=False):
    dets = _state["dets"];  stem = _state["stem"]
    if not stem: return
    mask = _include_mask()

    promoted = []
    for ci, cb in enumerate(_state.get("cand_checkboxes", [])):
        if cb.value and ci < len(_state["candidates"]):
            cand = dict(_state["candidates"][ci])
            cand.pop("_cand_idx", None)
            cand["source"] = "sam_candidate"
            cand["index"]  = sum(1 for inc in mask if inc) + len(promoted)
            promoted.append(cand)

    approved = []
    for d, inc in zip(dets, mask):
        if inc:
            d = dict(d)
            if "source" not in d: d["source"] = "sam_approved"
            approved.append(d)
    approved.extend(promoted)

    (ANNOTATED / f"{stem}_approved.json").write_text(json.dumps(approved, indent=2))
    _mark_clean()
    out_status.clear_output()
    with out_status:
        tag = "Auto-saved" if autosave else "Saved"
        print(f"{tag}: {len(approved)}/{len(dets)} approved → {stem}_approved.json")
        if promoted: print(f"  + {len(promoted)} promoted candidate(s)")


# ── Wire up observers ─────────────────────────────────────────────────────────
w_panel.observe(_load_panel, names="value")
w_contain.observe(_on_threshold, names="value")
w_min_iou.observe(_load_panel, names="value")
btn_save.on_click(_save)
btn_all_in.on_click(_include_all)
btn_all_out.on_click(_exclude_all)
btn_reset.on_click(_reset_to_auto)
btn_candidates.on_click(_on_candidates_toggle)
btn_add_manual.on_click(_on_add_manual)

# Draw sliders fire live preview on every move
for _w in (w_mx, w_my, w_mw, w_mh):
    _w.observe(_on_draw_change, names="value")


# ── Layout ────────────────────────────────────────────────────────────────────
display(
    widgets.HTML("<h3 style='margin:4px 0'>Bounding Box Review</h3>"),
    w_panel,
    widgets.HBox([w_contain, w_min_iou]),
    widgets.HBox([btn_all_in, btn_all_out, btn_reset, btn_save, btn_candidates]),
    out_status,
    out_panel,
    draw_section,
    widgets.HTML("<b style='font-size:13px'>Detection cards — toggle to include/exclude</b>"),
    out_cards,
    widgets.HTML("<b style='font-size:13px;color:#ffcc00'>"
                 "Candidate pool (SAM masks below filter threshold)</b>"),
    out_cands,
)
_load_panel()

HTML(value="<h3 style='margin:4px 0'>Bounding Box Review</h3>")

Dropdown(description='Panel:', layout=Layout(width='90%'), options=(('EBA-B_00425_Ibadan_q97912_i1_panel_00', …

Output()

Output()

HTML(value="<b style='font-size:13px'>Detection cards — toggle to include/exclude</b>")

Output()

HTML(value="<b style='font-size:13px;color:#ffcc00'>Candidate pool (SAM masks below filter threshold)</b>")

Output()

ValueError: Invalid format specifier '.3f if isinstance(area,float) else area' for object of type 'float'